In [ ]:
# This file is run on Google Colab GPU. 
# Colab setup — run this first
# torch / transformers are pre-installed on Colab, nothing extra needed

# Combination Model: FinBERT + Price Features → Tabular Transformer

**Architecture:** FT-Transformer (Feature Tokenizer + Transformer Encoder)  
Each of the 7 scalar features is embedded into a 32-dim vector, then processed by 2 Transformer encoder layers with self-attention.  
**Target:** intraday direction (close ≥ open), consistent with StockPredict 64.1% baseline.

In [4]:
import os
import shutil
from google.colab import files

WORK_DIR = "/content"
DATA_FILE = f"{WORK_DIR}/sp500_headlines_with_prices.csv"

# Upload data file (only needed once per session)
TARGET_NAME = "sp500_headlines_with_prices.csv"

if not os.path.exists(DATA_FILE):
    print(f"Please upload: {TARGET_NAME}")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = os.path.join(WORK_DIR, TARGET_NAME)
        shutil.move(filename, dest)
        print(f"Saved to {dest}")
else:
    print(f"{TARGET_NAME} already exists, skipping upload.")

Please upload: sp500_headlines_with_prices.csv


Saving sp500_headlines_with_prices.csv to sp500_headlines_with_prices.csv
Saved to /content/sp500_headlines_with_prices.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import os, gc, shutil
from google.colab import files

WORK_DIR   = "/content"
DATA_FILE  = f"{WORK_DIR}/sp500_headlines_with_prices.csv"
CACHE_FILE = f"{WORK_DIR}/finbert_probs_combo.npy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(42)
np.random.seed(42)

In [6]:
# 1. Load data & target
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["target"] = (df["close_price"] >= df["open_price"]).astype(int)
print(f"Total rows: {len(df)}")
print(f"Target distribution: {df['target'].value_counts().to_dict()}")

Total rows: 3507
Target distribution: {1: 1898, 0: 1609}


/tmp/ipykernel_815/3174075544.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


In [7]:
# 2. FinBERT sentiment (cached)
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert   = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
finbert.eval()

def get_finbert_probs(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt",
                           truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = finbert(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

if os.path.exists(CACHE_FILE):
    print("Loading cached FinBERT probs...")
    finbert_probs = np.load(CACHE_FILE)
else:
    print("Running FinBERT inference...")
    texts = df["combined_headline"].fillna("neutral").tolist()
    finbert_probs = get_finbert_probs(texts)
    np.save(CACHE_FILE, finbert_probs)
    print(f"Saved to {CACHE_FILE}")

# {0: positive, 1: negative, 2: neutral}
df["sent_pos"] = finbert_probs[:, 0]
df["sent_neg"] = finbert_probs[:, 1]
df["sent_neu"] = finbert_probs[:, 2]

# Free GPU/MPS memory before the rest
del finbert, tokenizer
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("FinBERT features added. Model freed from memory.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running FinBERT inference...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Saved to /content/finbert_probs_combo.npy
FinBERT features added. Model freed from memory.


In [8]:
# 3. Feature engineering — no lookahead leakage
# All features are computable at/before market open on day i:
#   sent_*        : today's news (available before open)
#   gap_open      : today's open vs yesterday's close (known at open)
#   prev_*        : yesterday's close-based indicators (fully historical)

df["gap_open"] = (df["open_price"] - df["close_price"].shift(1)) / df["close_price"].shift(1)

df["_ma5"]     = df["close_price"].rolling(5).mean()
df["_ma10"]    = df["close_price"].rolling(10).mean()
df["_vol_ma5"] = df["volume"].rolling(5).mean()

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs  = avg_gain / (avg_loss + 1e-9)
    return (100 - 100 / (1 + rs)) / 100.0

df["_rsi14"] = compute_rsi(df["close_price"], 14)

# Shift by 1: yesterday's values
df["prev_ma5_ratio"]  = df["close_price"].shift(1) / df["_ma5"].shift(1)
df["prev_rsi14"]      = df["_rsi14"].shift(1)
df["prev_vol_ratio"]  = df["volume"].shift(1) / df["_vol_ma5"].shift(1)

FEATURE_COLS = [
    "sent_pos", "sent_neg", "sent_neu",             # same-day FinBERT (3)
    "gap_open",                                      # open gap signal   (1)
    "prev_ma5_ratio", "prev_rsi14", "prev_vol_ratio" # yesterday tech    (3)
]

df = df.dropna(subset=FEATURE_COLS + ["target"]).reset_index(drop=True)
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Rows after dropna: {len(df)}")

Features (7): ['sent_pos', 'sent_neg', 'sent_neu', 'gap_open', 'prev_ma5_ratio', 'prev_rsi14', 'prev_vol_ratio']
Rows after dropna: 3492


In [9]:
# 4. Train / test split (chronological) + scale
split_index = int(len(df) * 0.8)

X_train = df[FEATURE_COLS].iloc[:split_index].values
y_train = df["target"].iloc[:split_index].values
X_test  = df[FEATURE_COLS].iloc[split_index:].values
y_test  = df["target"].iloc[split_index:].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Test target dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Train: (2793, 7) | Test: (699, 7)
Test target dist: {np.int64(0): np.int64(330), np.int64(1): np.int64(369)}


In [ ]:
# 5. Tabular Transformer — architecture
# Each scalar feature → independent 32-dim linear embedding (FT-Transformer style)
# CLS token prepended; 2-layer Transformer encoder; CLS output → binary classifier

D_MODEL      = 32
N_HEADS      = 2
N_LAYERS     = 2
FFN_DIM      = 64
DROPOUT      = 0.3
LR           = 3e-4
WEIGHT_DECAY = 1e-3
BATCH_SIZE   = 64
EPOCHS       = 150
PATIENCE     = 25

class FeatureTokenizer(nn.Module):
    """Each scalar feature x_i → x_i * w_i + b_i  (per-feature linear, shape → d_model)"""
    def __init__(self, n_features, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_model))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.weight, a=0.01)

    def forward(self, x):
        # x: (B, n_features)  →  (B, n_features, d_model)
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class TabTransformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_dim, dropout):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True,  # pre-norm: more stable on small data
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        tokens = self.tokenizer(x)                              # (B, F, d_model)
        cls    = self.cls_token.expand(x.size(0), -1, -1)      # (B, 1, d_model)
        tokens = torch.cat([cls, tokens], dim=1)                # (B, F+1, d_model)
        out    = self.encoder(tokens)
        return self.head(self.norm(out[:, 0])).squeeze(1)       # CLS output → logit

model = TabTransformer(
    n_features=len(FEATURE_COLS),
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    ffn_dim=FFN_DIM, dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
# 6. Training loop with early stopping
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)

criterion = nn.BCEWithLogitsLoss()   # classes ~balanced (53/47), no pos_weight needed
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float("inf")
patience_ctr  = 0
best_state    = None

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(X_tr)
    scheduler.step()

    # --- validate ---
    model.eval()
    with torch.no_grad():
        val_logits = model(X_te.to(device))
        val_loss   = criterion(val_logits, y_te.to(device)).item()
        val_preds  = (torch.sigmoid(val_logits) > 0.5).cpu().int().numpy()
        val_acc    = accuracy_score(y_test, val_preds)

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (patience={PATIENCE})")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# 7. Evaluate best model
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    logits = model(X_te.to(device))
    probs  = torch.sigmoid(logits).cpu().numpy()
    preds  = (probs > 0.5).astype(int)

print("=== Transformer (FinBERT + Price) ===")
print(classification_report(y_test, preds))
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, probs), 4))

majority = y_test.sum() / len(y_test)
print("\n============================================")
print(f"  Random baseline (always up)          : {majority:.1%}")
print(f"  StockPredict   FinBERT→LinearSVC     : 64.1%")
print(f"  Combination    LinearSVC (7 features): 64.2%")
print(f"  Combination    Transformer            : {accuracy_score(y_test, preds):.1%}")
print("============================================")

In [ ]:
# 8. Feature importance via input-gradient saliency
# grad * input magnitude tells us which features most influenced the predictions
model.eval()
X_sal = X_te.to(device).requires_grad_(True)
model(X_sal).sum().backward()

saliency = (X_sal.grad.abs() * X_sal.detach().abs()).mean(0).cpu().numpy()
fi = pd.Series(saliency, index=FEATURE_COLS).sort_values(ascending=False)
print("--- Feature saliency (gradient × input) ---")
print(fi.to_string())